# Bottom temperature and salinity bias in ACCESS-OM3

Bias in **bottom conservative temperature** and **bottom absolute salinity** over the whole Southern Ocean floor (south of 45°S), compared with the OCEAN ICE Southern Ocean Climatology (post-2000 version):

> Zhou, S., Dutrieux, P., Giulivi, C., et al. *The OCEAN ICE hydrography profiles compilation and climatology*, ESSD Discussions, https://doi.org/10.5194/essd-2025-727. Data: https://doi.org/10.17882/103946 (file `OI_2000_Climatology.nc`, annual mean built from post-2000 profiles, CC-BY 4.0).

The climatology resolves the Antarctic continental shelf much better than WOA (averaging length scales down to 20 km on the shelf), so it is well suited to evaluating Dense Shelf Water and Antarctic Bottom Water properties. ACCESS-OM2-01 (IAF cycle 1) is shown alongside for comparison.

**Salinity / temperature scales — read this before changing anything.** The climatology is on the TEOS-10 scale (conservative temperature Θ in °C, absolute salinity S_A in g/kg). ACCESS-OM3 runs with `USE_CONTEMP_ABSSAL = True`, so its prognostic variables are Θ and S_A, **but** the saved `so` diagnostic is *practical* salinity (units `psu`). Comparing `so` directly with S_A gives a spurious ≈ −0.16 g/kg "bias", which is the same size as the shelf biases we are looking for. So:
- OM3 temperature: `bigthetao` (conservative temperature, °C), *not* `thetao` (potential temperature).
- OM3 salinity: `so` (practical) → converted to S_A with `gsw.SA_from_SP`.
- OM2-01 temperature: `temp` is conservative temperature in **K** → subtract 273.15.
- OM2-01 salinity: `salt` (practical) → converted to S_A with `gsw.SA_from_SP`.

In [ ]:
#parameters

# These first two cells must be in all notebooks!
# It allows us to run all the notebooks at once, this cell has a tag "parameters" which allows us to pass in 
# arguments externally using papermill (see mkfigs.sh for details)

# Set esm_file to the datastore for the main experiment of interest
esm_file = "/g/data/ol01/outputs/access-om3-25km/MC_25km_jra_iaf+wombatlite-test5.1-8f50b769/datastore.json"
# esm_file = "/g/data/zv30/non-cmip/ACCESS-CM3/cm3-run-27-07-2026-PD-control/cm3-datastore/cm3-datastore.json"

# papermill settings. *No need to modify these if running interactively.* 
papermill = False                      # `cwd` and `nbname` will be populated by papermill.
cwd = None                             # current working directory 
nbname = None                          # notebook name

In [ ]:
if not papermill: 
    import nci_ipynb, os  # requires conda/analysis3-26.03 or later
    cwd = nci_ipynb.dir()
    nbname = nci_ipynb.name()
    os.chdir(cwd)
import mkfigs_bootstrap  # noqa: adds external/access-model-mkfigs/src to sys.path (stop-gap)
from mkfigs import MkmdWriter
mkmd = MkmdWriter(esm_file, nbname, str(cwd), pm=papermill)
from model_agnostic import patch_broken_conda_env
from exptdata_access import get_exptname_from_path
patch_broken_conda_env()  # conda env workaround for analysis3-26.03+; no-op on healthy envs
from exptdata_access import guess_experiment_from_esm_file
expt_key, info = guess_experiment_from_esm_file(esm_file)
model_name = info["model"]  # OM3 or CM3

In [ ]:
import os
plotfolder = str(cwd) + "/" if cwd else "./"
os.makedirs(plotfolder, exist_ok=True)
notebook_name = "Bottom_TS_bias"

In [ ]:
import intake
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.path as mpath
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cmocean as cm
import gsw  # TEOS-10 Gibbs SeaWater toolbox, for practical -> absolute salinity
from distributed import Client

# Stop annoying warnings coming out of xarray.
import warnings
warnings.filterwarnings('ignore')

In [ ]:
client = Client(threads_per_worker=1)
from model_agnostic import patch_dask_workers
patch_dask_workers(client)  # patch workers too
print(client.dashboard_link)

## Choose the averaging period

The climatology is built from post-2000 profiles. ACCESS-OM2-01 IAF cycle 1 ends in 2018, so both models are averaged over the same years inside 2000–2018. The default is the last decade of that window; widen it if you like, but the OM2-01 read gets expensive (0.1° monthly 3D fields).

In [ ]:
start_year = 2009
end_year   = 2018
time_window = slice(f"{start_year}-01-01", f"{end_year}-12-31")

northern_limit = -45  # the climatology covers 90°S–45°S only

User should not need to adjust anything in code below.

## Load the OCEAN ICE climatology and extract the bottom value

The file is a single annual-mean 3D field on a regular 0.2° (lon) × 0.1° (lat) grid with 10 dbar vertical spacing. Its "bottom" value is the deepest level in each column that holds data.

In [ ]:
obs_file = "/g/data/v46/txs156/obs/OCEAN_ICE_climatology/OI_2000_Climatology.nc"

obs = xr.open_dataset(obs_file)

# The file stores the coordinates as 2D (with a dummy SCALAR dimension) and calls the dimensions nx/ny/nz.
# Turn them into normal 1D coordinates called lon/lat/pressure so xarray can select and interpolate.
obs = obs.assign_coords(
    lon      = ("nx", obs.longitude.values.squeeze()),
    lat      = ("ny", obs.latitude.values.squeeze()),
    pressure = ("nz", obs.pressure.values.squeeze()),
).swap_dims({"nx": "lon", "ny": "lat", "nz": "pressure"})
obs = obs[["ct", "sa"]].sel(lat=slice(-90, northern_limit))

# Check the units really are TEOS-10 (conservative temperature, absolute salinity) before using them
for var in ["ct", "sa"]:
    print(var, "|", obs[var].attrs.get("standard name"), "|", obs[var].attrs.get("unit"))
obs

In [ ]:
# Load into memory (~3 GB per variable south of 45°S) so the bottom search is quick
obs_ct = obs.ct.load()   # dims: (pressure, lon, lat)
obs_sa = obs.sa.load()

# Index of the deepest level that has data in each column.
# has_data is True where the climatology is defined; multiply by the level number and take the max.
has_data = obs_ct.notnull()
level_number = xr.DataArray(np.arange(obs_ct.sizes["pressure"]), dims="pressure")
bottom_level = (has_data * level_number).max("pressure")
column_has_any_data = has_data.any("pressure")

# Pick out the bottom value in each column; columns with no data at all (land) become NaN
obs_ct_bottom = obs_ct.isel(pressure=bottom_level).where(column_has_any_data)
obs_sa_bottom = obs_sa.isel(pressure=bottom_level).where(column_has_any_data)
obs_bottom_pressure = obs_ct.pressure.isel(pressure=bottom_level).where(column_has_any_data)

# Put lat first, like the model fields: dims (lat, lon)
obs_ct_bottom = obs_ct_bottom.transpose("lat", "lon")
obs_sa_bottom = obs_sa_bottom.transpose("lat", "lon")
obs_bottom_pressure = obs_bottom_pressure.transpose("lat", "lon")

## Load ACCESS-OM3 and extract the bottom value

Bottom value = the deepest wet `z_l` cell in each column (the same method as `Bottom_age_tracer_in_ACCESS_OM3.ipynb`). The time mean is taken first; the wet/dry pattern of `z_l` cells doesn't change in time, so this gives the same answer as taking the bottom of each month.

In [ ]:
datastore = intake.open_esm_datastore(
    esm_file,
    columns_with_iterables=[
        "variable",
        "variable_long_name",
        "variable_standard_name",
        "variable_cell_methods",
        "variable_units"
    ]
)

In [ ]:
# Conservative temperature (bigthetao) and *practical* salinity (so), monthly, south of 45°S
OM3_files_kwargs = dict(chunks={"z_l": -1, "yh": -1, "xh": -1}, decode_timedelta=True)

ct_OM3 = datastore.search(variable="bigthetao", frequency="1mon").to_dask(xarray_open_kwargs=OM3_files_kwargs).bigthetao
sp_OM3 = datastore.search(variable="so",        frequency="1mon").to_dask(xarray_open_kwargs=OM3_files_kwargs).so

# Check the units: bigthetao should be conservative temperature in Celsius, so should be practical salinity in psu
for da in [ct_OM3, sp_OM3]:
    print(da.name, "|", da.attrs.get("standard_name"), "|", da.attrs.get("units"), "|", da.attrs.get("long_name"))

ct_OM3 = ct_OM3.sel(time=time_window, yh=slice(-90, northern_limit))
sp_OM3 = sp_OM3.sel(time=time_window, yh=slice(-90, northern_limit))
print("OM3 months averaged:", ct_OM3.sizes["time"], "from", str(ct_OM3.time.values[0])[:10], "to", str(ct_OM3.time.values[-1])[:10])

In [ ]:
# Time mean over the averaging period
ct_OM3_mean = ct_OM3.mean("time").load()   # dims: (z_l, yh, xh)
sp_OM3_mean = sp_OM3.mean("time").load()

# Depth of every wet cell (NaN where the cell is land/below the sea floor), then the deepest wet depth per column
depth_array_OM3 = ct_OM3_mean * 0 + ct_OM3_mean.z_l
max_depth_OM3   = depth_array_OM3.max(dim="z_l", skipna=True)

# Keep only the deepest wet cell and collapse the vertical (min_count=1 keeps land as NaN instead of 0)
is_bottom_cell_OM3 = depth_array_OM3 >= max_depth_OM3
ct_bottom_OM3 = ct_OM3_mean.where(is_bottom_cell_OM3).sum("z_l", min_count=1)
sp_bottom_OM3 = sp_OM3_mean.where(is_bottom_cell_OM3).sum("z_l", min_count=1)

### Convert OM3 bottom salinity from practical to absolute salinity

`gsw.SA_from_SP(SP, p, lon, lat)` needs sea pressure (dbar) at the bottom cell, from its depth with `gsw.p_from_z`. South of 45°S the OM3 grid is a regular Mercator grid, so the 1D `xh`/`yh` coordinates are the true longitude/latitude there.

In [ ]:
lon_OM3, lat_OM3 = xr.broadcast(ct_bottom_OM3.xh, ct_bottom_OM3.yh)
pressure_bottom_OM3 = gsw.p_from_z(-max_depth_OM3, lat_OM3)   # z is negative downwards in gsw

sa_bottom_OM3 = gsw.SA_from_SP(sp_bottom_OM3, pressure_bottom_OM3, lon_OM3, lat_OM3)
sa_bottom_OM3 = sa_bottom_OM3.transpose("yh", "xh")
ct_bottom_OM3 = ct_bottom_OM3.transpose("yh", "xh")

### Interpolate the climatology bottom values onto the OM3 grid and compute the bias

The climatology grid (0.2° × 0.1°) is finer than OM3-25km, so it is bilinearly interpolated onto the OM3 cell centres; the bias is model − observations. OM3 longitudes run from −280 to 80, so they are wrapped onto the climatology's −180…180 first.

Caveat: the model and the climatology have different bathymetry, so the "bottom" is not always at the same depth (especially on the slope). Very steep regions should be interpreted with care.

In [ ]:
lon_OM3_wrapped = ((ct_bottom_OM3.xh + 180) % 360) - 180

obs_ct_bottom_on_OM3 = obs_ct_bottom.interp(lat=ct_bottom_OM3.yh, lon=lon_OM3_wrapped)
obs_sa_bottom_on_OM3 = obs_sa_bottom.interp(lat=ct_bottom_OM3.yh, lon=lon_OM3_wrapped)

ct_bias_OM3 = ct_bottom_OM3 - obs_ct_bottom_on_OM3
sa_bias_OM3 = sa_bottom_OM3 - obs_sa_bottom_on_OM3

## Load ACCESS-OM2-01 (IAF cycle 1) and extract the bottom value

In [ ]:
catalog = intake.cat.access_nri
expt_OM2_01 = "01deg_jra55v140_iaf"  # cycle 1 of OM2-01 IAF simulation

# OM2-01 saves both monthly means and (from 2012) monthly snapshots of temp/salt; temporal_label="mean" picks the means
# A full 0.1° 3D field is ~3 GB, too big for one dask worker, so read it in pieces that match the on-disk chunks
OM2_files_kwargs = dict(chunks={"time": 1, "st_ocean": 7, "yt_ocean": 300, "xt_ocean": 400}, decode_timedelta=True)

temp_OM2_01 = catalog[expt_OM2_01].search(variable="temp", frequency="1mon", temporal_label="mean").to_dask(xarray_open_kwargs=OM2_files_kwargs).temp
salt_OM2_01 = catalog[expt_OM2_01].search(variable="salt", frequency="1mon", temporal_label="mean").to_dask(xarray_open_kwargs=OM2_files_kwargs).salt

# Check the units: temp should be conservative temperature in K, salt practical salinity in psu
for da in [temp_OM2_01, salt_OM2_01]:
    print(da.name, "|", da.attrs.get("standard_name"), "|", da.attrs.get("units"), "|", da.attrs.get("long_name"))

temp_OM2_01 = temp_OM2_01.sel(time=time_window, yt_ocean=slice(-90, northern_limit))
salt_OM2_01 = salt_OM2_01.sel(time=time_window, yt_ocean=slice(-90, northern_limit))
print("OM2-01 months averaged:", temp_OM2_01.sizes["time"], "from", str(temp_OM2_01.time.values[0])[:10], "to", str(temp_OM2_01.time.values[-1])[:10])

In [ ]:
# Time mean; convert conservative temperature from Kelvin to Celsius
ct_OM2_01_mean = temp_OM2_01.mean("time").load() - 273.15   # dims: (st_ocean, yt_ocean, xt_ocean)
sp_OM2_01_mean = salt_OM2_01.mean("time").load()

# Same bottom-cell method as for OM3
depth_array_OM2_01 = ct_OM2_01_mean * 0 + ct_OM2_01_mean.st_ocean
max_depth_OM2_01   = depth_array_OM2_01.max(dim="st_ocean", skipna=True)

is_bottom_cell_OM2_01 = depth_array_OM2_01 >= max_depth_OM2_01
ct_bottom_OM2_01 = ct_OM2_01_mean.where(is_bottom_cell_OM2_01).sum("st_ocean", min_count=1)
sp_bottom_OM2_01 = sp_OM2_01_mean.where(is_bottom_cell_OM2_01).sum("st_ocean", min_count=1)

# Practical -> absolute salinity (OM2-01 is regular Mercator south of 65°N, so xt_ocean/yt_ocean are true lon/lat)
lon_OM2_01, lat_OM2_01 = xr.broadcast(ct_bottom_OM2_01.xt_ocean, ct_bottom_OM2_01.yt_ocean)
pressure_bottom_OM2_01 = gsw.p_from_z(-max_depth_OM2_01, lat_OM2_01)
sa_bottom_OM2_01 = gsw.SA_from_SP(sp_bottom_OM2_01, pressure_bottom_OM2_01, lon_OM2_01, lat_OM2_01)

sa_bottom_OM2_01 = sa_bottom_OM2_01.transpose("yt_ocean", "xt_ocean")
ct_bottom_OM2_01 = ct_bottom_OM2_01.transpose("yt_ocean", "xt_ocean")

In [ ]:
# Interpolate the climatology onto the OM2-01 grid (OM2-01 longitudes also run from -280 to 80) and compute the bias
lon_OM2_01_wrapped = ((ct_bottom_OM2_01.xt_ocean + 180) % 360) - 180

obs_ct_bottom_on_OM2_01 = obs_ct_bottom.interp(lat=ct_bottom_OM2_01.yt_ocean, lon=lon_OM2_01_wrapped)
obs_sa_bottom_on_OM2_01 = obs_sa_bottom.interp(lat=ct_bottom_OM2_01.yt_ocean, lon=lon_OM2_01_wrapped)

ct_bias_OM2_01 = ct_bottom_OM2_01 - obs_ct_bottom_on_OM2_01
sa_bias_OM2_01 = sa_bottom_OM2_01 - obs_sa_bottom_on_OM2_01

## Plotting

South Polar Stereographic maps of 45°S–90°S. Rows: observed bottom value, OM3 − obs, OM2-01 − obs. Columns: bottom conservative temperature, bottom absolute salinity. The black contour is the model's 1000 m isobath (OM3 isobath on the observation and OM3 panels, OM2-01 isobath on the OM2-01 panels).

In [ ]:
# Bathymetry for the 1000 m isobath
deptho_search = datastore.search(variable="deptho")
deptho_OM3 = deptho_search.search(path=deptho_search.df.path[0]).to_dask()["deptho"].sel(yh=slice(-90, northern_limit)).load()
ht_OM2_01 = catalog[expt_OM2_01].search(variable="ht").to_dask()["ht"].sel(yt_ocean=slice(-90, northern_limit)).load()
if "time" in ht_OM2_01.dims:
    ht_OM2_01 = ht_OM2_01.isel(time=0)

om3_res = {360: "100km", 1440: "25km", 4320: "8km"}[deptho_OM3.sizes["xh"]]
ft_size = 12  # fontsize

# Circular boundary for the polar stereographic maps
theta = np.linspace(0, 2 * np.pi, 100)
circle = mpath.Path(np.vstack([np.sin(theta), np.cos(theta)]).T * 0.5 + 0.5)

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(11, 15), subplot_kw={"projection": ccrs.SouthPolarStereo()})

# Colour limits: absolute values on the top row, symmetric bias limits on the bottom two rows
ct_levels, sa_levels = (-2, 2), (34.6, 35.0)
ct_bias_lim, sa_bias_lim = 1.0, 0.2

panels = [
    # (row, col, field, lon, lat, cmap, vmin, vmax, colour-bar label, title, isobath bathymetry)
    (0, 0, obs_ct_bottom, obs_ct_bottom.lon, obs_ct_bottom.lat, cm.cm.thermal, *ct_levels,  "Θ (°C)",    "OCEAN ICE (post-2000) bottom Θ", deptho_OM3),
    (0, 1, obs_sa_bottom, obs_sa_bottom.lon, obs_sa_bottom.lat, cm.cm.haline,  *sa_levels,  "S$_A$ (g/kg)", "OCEAN ICE (post-2000) bottom S$_A$", deptho_OM3),
    (1, 0, ct_bias_OM3, ct_bias_OM3.xh, ct_bias_OM3.yh, cm.cm.balance, -ct_bias_lim, ct_bias_lim, "ΔΘ (°C)", f"ACCESS-OM3-{om3_res} − obs, bottom Θ", deptho_OM3),
    (1, 1, sa_bias_OM3, sa_bias_OM3.xh, sa_bias_OM3.yh, cm.cm.balance, -sa_bias_lim, sa_bias_lim, "ΔS$_A$ (g/kg)", f"ACCESS-OM3-{om3_res} − obs, bottom S$_A$", deptho_OM3),
    (2, 0, ct_bias_OM2_01, ct_bias_OM2_01.xt_ocean, ct_bias_OM2_01.yt_ocean, cm.cm.balance, -ct_bias_lim, ct_bias_lim, "ΔΘ (°C)", "ACCESS-OM2-01 − obs, bottom Θ", ht_OM2_01),
    (2, 1, sa_bias_OM2_01, sa_bias_OM2_01.xt_ocean, sa_bias_OM2_01.yt_ocean, cm.cm.balance, -sa_bias_lim, sa_bias_lim, "ΔS$_A$ (g/kg)", "ACCESS-OM2-01 − obs, bottom S$_A$", ht_OM2_01),
]

panel_letters = "abcdef"
for i, (row, col, field, lon, lat, cmap, vmin, vmax, label, title, bathy) in enumerate(panels):
    ax = axes[row, col]
    ax.set_extent([-180, 180, -90, northern_limit], crs=ccrs.PlateCarree())
    ax.set_boundary(circle, transform=ax.transAxes)
    ax.add_feature(cfeature.LAND, color="darkgrey", zorder=0)

    im = ax.pcolormesh(lon, lat, field, cmap=cmap, vmin=vmin, vmax=vmax, transform=ccrs.PlateCarree(), rasterized=True)

    # 1000 m isobath of the model grid that matches the panel
    bathy_lon = bathy[bathy.dims[-1]]
    bathy_lat = bathy[bathy.dims[-2]]
    ax.contour(bathy_lon, bathy_lat, bathy, levels=[1000], colors="black", linewidths=0.8, transform=ccrs.PlateCarree())

    ax.gridlines(color="grey", alpha=0.4, linestyle="--")
    plt.colorbar(im, ax=ax, shrink=0.75, extend="both", label=label)
    ax.set_title(title, fontsize=ft_size)
    ax.set_title(panel_letters[i], fontweight="bold", fontsize=ft_size, loc="left")

fig.suptitle(f"Bottom Θ and S$_A$, model mean {start_year}–{end_year}\n{get_exptname_from_path(esm_file)}", fontsize=ft_size)
fig.tight_layout()

mkmd.savefig(fig, "Bottom temperature and salinity bias",
             f"Bottom conservative temperature (left) and absolute salinity (right) south of 45°S. "
             f"(a,b) OCEAN ICE post-2000 climatology (Zhou et al., doi:10.17882/103946); "
             f"(c,d) ACCESS-OM3-{om3_res} minus climatology; (e,f) ACCESS-OM2-01 (01deg_jra55v140_iaf) minus climatology. "
             f"Models averaged over {start_year}–{end_year}; model practical salinity converted to absolute salinity with TEOS-10. "
             f"Black contour: model 1000 m isobath.", dpi=150)

In [ ]:
# Area-weighted mean bias on the continental shelf (shallower than 1000 m) and in the deep ocean, OM3 only.
# Only cells where both the model and the climatology have data are used.
areacello = datastore.search(variable="areacello").to_dask()["areacello"]
if "time" in areacello.dims:
    areacello = areacello.isel(time=0)
areacello = areacello.sel(yh=slice(-90, northern_limit)).load()

on_shelf = deptho_OM3 < 1000
summary = {}
for name, bias in [("Θ bias (°C)", ct_bias_OM3), ("S_A bias (g/kg)", sa_bias_OM3)]:
    valid = bias.notnull()
    summary[name] = {
        "shelf (<1000 m)": float(bias.where(on_shelf & valid).weighted(areacello.fillna(0)).mean()),
        "deep (≥1000 m)":  float(bias.where(~on_shelf & valid).weighted(areacello.fillna(0)).mean()),
    }

import pandas as pd
summary_table = pd.DataFrame(summary).T.round(3)
summary_table

In [ ]:
client.close()